# Module 03, Notebook 1: Synthetic Control Basics

## Learning Objectives
By completing this notebook, you will:
1. Build a panel dataset suitable for synthetic control analysis
2. Formulate and solve the weight optimization problem using scipy
3. Visualize pre-period fit and post-period counterfactual gap
4. Compute the estimated causal effect with uncertainty bounds
5. Interpret donor weight sparsity as a diagnostic

## The Synthetic Control Idea

Instead of extrapolating the treated unit's own pre-trend, synthetic control constructs a counterfactual
from a weighted average of untreated donor units. The weights are chosen to minimize the pre-intervention
discrepancy between the treated unit and the weighted donor average.

We simulate a tobacco-control study modeled on the California Prop 99 (Abadie et al., 2010).
One state implements a cigarette tax; 20 donor states do not.

## Estimated Time: 15 minutes

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

## 1. Generate the Panel Dataset

We simulate cigarette consumption (packs per capita per year) for 21 states over 30 years.
State 0 is California. In year 20, California implements a tobacco tax (−20 packs/year immediate drop,
−0.5 packs/year further slope change).

Donor states share similar pre-period trends but are not affected by the California policy.

In [ ]:
# --- Dataset parameters ---
N_YEARS = 30
N_PRE = 20         # Pre-intervention: years 0-19
N_POST = 10        # Post-intervention: years 20-29
N_DONORS = 20      # Donor states
T_STAR = N_PRE     # Intervention at year 20

TRUE_LEVEL_EFFECT = -20.0  # Immediate drop in packs/year
TRUE_SLOPE_EFFECT = -0.5   # Additional annual decline post-intervention

years = np.arange(N_YEARS)

# --- Generate donor trajectories ---
# Each donor has: baseline, shared trend, unit-specific trend, and idiosyncratic noise
donor_baselines = np.random.uniform(100, 140, N_DONORS)
shared_trend = -1.2  # All states share declining trend (secular decline in smoking)
donor_own_trends = np.random.normal(0, 0.3, N_DONORS)  # Unit-specific variation

Y_donors = np.zeros((N_YEARS, N_DONORS))
for j in range(N_DONORS):
    trend = shared_trend + donor_own_trends[j]
    Y_donors[:, j] = (
        donor_baselines[j]
        + trend * years
        + np.random.normal(0, 3.0, N_YEARS)
    )

# --- Generate California trajectory ---
# California is a weighted combination of specific donors + noise
# This ensures good pre-period fit is achievable
TRUE_WEIGHTS = np.zeros(N_DONORS)
TRUE_WEIGHTS[0] = 0.40   # Colorado
TRUE_WEIGHTS[3] = 0.35   # Utah
TRUE_WEIGHTS[7] = 0.25   # Montana

# California pre-period: weighted sum of donors + small noise
y_california_latent = Y_donors @ TRUE_WEIGHTS + np.random.normal(0, 1.5, N_YEARS)

# Post-intervention: add the treatment effect
y_california = y_california_latent.copy()
for t in range(T_STAR, N_YEARS):
    time_post = t - T_STAR  # 0 in year T_STAR, 1 in year T_STAR+1, ...
    y_california[t] += TRUE_LEVEL_EFFECT + TRUE_SLOPE_EFFECT * time_post

print(f"California pre-period mean:  {y_california[:N_PRE].mean():.1f} packs/year")
print(f"California post-period mean: {y_california[N_PRE:].mean():.1f} packs/year")
print(f"Naive pre-post difference:   {y_california[N_PRE:].mean() - y_california[:N_PRE].mean():.1f}")
print()
print("True treatment effects:")
print(f"  Immediate level change: {TRUE_LEVEL_EFFECT} packs/year")
print(f"  Additional slope change: {TRUE_SLOPE_EFFECT} packs/year per year")

## 2. Visualize the Raw Data

Before fitting the synthetic control, look at the raw data. California's trajectory should be
visible among the donor bundle, with a clear break at year 20.

In [ ]:
# Raw trajectory plot: all states

fig, ax = plt.subplots(figsize=(12, 6))

# Donor states: light gray
for j in range(N_DONORS):
    ax.plot(years, Y_donors[:, j], color="#aaaaaa", alpha=0.4, linewidth=1)

# California: bold blue
ax.plot(years, y_california, color="#2980b9", linewidth=2.5, label="California (treated)")

ax.axvline(T_STAR, color="#e74c3c", linestyle="--", linewidth=2, label="Tobacco tax (Year 20)")
ax.set_title("Raw Data: Cigarette Consumption — California vs Donor States", fontsize=13)
ax.set_xlabel("Year")
ax.set_ylabel("Packs per Capita per Year")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Observation: California's trajectory diverges downward from the donor bundle after Year 20.")
print("But how much of this is the tax vs the ongoing secular decline?")
print("The synthetic control will isolate the tax effect.")

## 3. Solve for Optimal Donor Weights

The optimization problem: find weights $\mathbf{w}$ with $w_j \geq 0$, $\sum_j w_j = 1$, that
minimize the pre-period discrepancy between California and the weighted donor average.

$$\mathbf{w}^* = \arg\min_\mathbf{w} \sum_{t=0}^{T_0-1} \left( Y_{\text{CA},t} - \sum_j w_j Y_{j,t} \right)^2$$

In [ ]:
# Solve the constrained quadratic optimization using scipy

y_ca_pre = y_california[:N_PRE]           # California pre-period: shape (N_PRE,)
Y_donors_pre = Y_donors[:N_PRE, :]        # Donors pre-period: shape (N_PRE, N_DONORS)
Y_donors_post = Y_donors[N_PRE:, :]       # Donors post-period: shape (N_POST, N_DONORS)


def objective(w, y_treated, Y_donors):
    """Squared distance between treated and weighted donor sum."""
    y_synthetic = Y_donors @ w
    return np.sum((y_treated - y_synthetic) ** 2)


# Constraints: weights sum to 1
constraints = [{"type": "eq", "fun": lambda w: w.sum() - 1.0}]

# Bounds: each weight between 0 and 1
bounds = [(0.0, 1.0)] * N_DONORS

# Initial guess: equal weights
w0 = np.ones(N_DONORS) / N_DONORS

result = minimize(
    objective,
    w0,
    args=(y_ca_pre, Y_donors_pre),
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={"maxiter": 2000, "ftol": 1e-12},
)

w_opt = result.x
print(f"Optimization converged: {result.success}")
print(f"Final objective value: {result.fun:.4f}")
print()

# Show non-trivial weights
state_names = [f"Donor_{j:02d}" for j in range(N_DONORS)]
state_names[0] = "Colorado"
state_names[3] = "Utah"
state_names[7] = "Montana"

weight_df = pd.DataFrame({"State": state_names, "Weight": w_opt}).sort_values("Weight", ascending=False)
print("Donor Weights (non-zero):")
print(weight_df[weight_df["Weight"] > 0.005].to_string(index=False, float_format="{:.3f}".format))
print()
print("True weights used to generate data:")
print(f"  Colorado (Donor_00): {TRUE_WEIGHTS[0]:.2f}")
print(f"  Utah    (Donor_03): {TRUE_WEIGHTS[3]:.2f}")
print(f"  Montana (Donor_07): {TRUE_WEIGHTS[7]:.2f}")

## 4. Compute Pre-Period Fit Quality

The synthetic control is credible only if the pre-period fit is good. Compute the RMSPE
(root mean squared prediction error) and visualize the pre-period trajectory.

In [ ]:
# Pre-period synthetic control trajectory
y_synthetic_pre = Y_donors_pre @ w_opt
y_synthetic_post = Y_donors_post @ w_opt

# Full synthetic trajectory
y_synthetic_full = np.concatenate([y_synthetic_pre, y_synthetic_post])

# Pre-period fit quality
pre_residuals = y_ca_pre - y_synthetic_pre
rmspe_pre = np.sqrt(np.mean(pre_residuals ** 2))
rmspe_threshold = 0.20 * y_ca_pre.std()  # 20% of pre-period std

print("Pre-Period Fit Quality")
print(f"  RMSPE:          {rmspe_pre:.2f} packs/year")
print(f"  20% threshold:  {rmspe_threshold:.2f} packs/year")
print(f"  Pre-period std: {y_ca_pre.std():.2f} packs/year")
print()
if rmspe_pre < rmspe_threshold:
    print("  GOOD FIT: RMSPE below 20% threshold — synthetic control is credible.")
else:
    print("  POOR FIT: RMSPE above 20% threshold — reconsider donor pool or predictors.")

# R-squared for pre-period
ss_res = np.sum(pre_residuals ** 2)
ss_tot = np.sum((y_ca_pre - y_ca_pre.mean()) ** 2)
r_squared_pre = 1 - ss_res / ss_tot
print(f"  R-squared (pre-period): {r_squared_pre:.4f}")

In [ ]:
# Visualize: California vs Synthetic California

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full trajectory
ax = axes[0]
ax.plot(years, y_california, color="#2980b9", linewidth=2.5, label="California (observed)")
ax.plot(years, y_synthetic_full, color="#e74c3c", linewidth=2.5,
        linestyle="--", label="Synthetic California")
ax.axvline(T_STAR, color="#27ae60", linestyle=":", linewidth=2, label="Tobacco tax")
ax.set_title("California vs Synthetic California", fontsize=12)
ax.set_xlabel("Year")
ax.set_ylabel("Packs per Capita per Year")
ax.legend(fontsize=10)

# Right: gap (causal effect estimate)
ax = axes[1]
gap_pre = y_ca_pre - y_synthetic_pre
gap_post = y_california[N_PRE:] - y_synthetic_post
gap_full = np.concatenate([gap_pre, gap_post])

ax.fill_between(years[N_PRE:], gap_post, 0, alpha=0.3, color="#e74c3c",
                label="Causal effect (gap)")
ax.plot(years[:N_PRE], gap_pre, color="#aaaaaa", linewidth=1.5, label="Pre-period gap")
ax.plot(years[N_PRE:], gap_post, color="#e74c3c", linewidth=2.5, label="Post-period gap")
ax.axhline(0, color="black", linestyle="--", linewidth=1.5)
ax.axvline(T_STAR, color="#27ae60", linestyle=":", linewidth=2)
ax.set_title("Causal Effect: California − Synthetic California", fontsize=12)
ax.set_xlabel("Year")
ax.set_ylabel("Gap (Packs per Capita per Year)")
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("The pre-period gap (gray) should be near zero — it validates the synthetic control.")
print("The post-period gap (red) is the estimated causal effect of the tobacco tax.")

## 5. Effect Size: Pointwise and Cumulative

Compute the causal effect at each post-intervention year and the cumulative effect over the
full post-intervention period.

In [ ]:
# Pointwise causal effect (gap at each post-period year)
post_years = years[N_PRE:]

# True effects at each post-period year
true_effects = np.array([
    TRUE_LEVEL_EFFECT + TRUE_SLOPE_EFFECT * t for t in range(N_POST)
])

print("Pointwise Causal Effect Estimates")
print(f"{'Year':<6} {'Estimated Gap':>16} {'True Effect':>14}")
print("-" * 40)
for i, year in enumerate(post_years):
    print(f"  {year:<4}  {gap_post[i]:>+14.1f}  {true_effects[i]:>+12.1f}")

print()
print("Cumulative Effect (total packs per capita)")
print(f"  Estimated cumulative gap: {gap_post.sum():.1f}")
print(f"  True cumulative effect:   {true_effects.sum():.1f}")
print(f"  Estimation error:         {gap_post.sum() - true_effects.sum():.1f}")

## 6. Bootstrap Uncertainty (Simple Approach)

The classical synthetic control does not have a parametric uncertainty estimate. A simple
bootstrap over time (resampling pre-period years) provides approximate uncertainty for the
donor weights. Note: the proper inference method for synthetic control is permutation tests
(covered in Notebook 02).

This bootstrap illustrates the concept — use permutation tests for formal inference.

In [ ]:
# Bootstrap over pre-period years to get weight uncertainty
# This is illustrative only — permutation tests (Notebook 02) are the valid inference method

N_BOOTSTRAP = 500
bootstrap_gaps = np.zeros((N_BOOTSTRAP, N_POST))

for b in range(N_BOOTSTRAP):
    # Resample pre-period years with replacement
    boot_idx = np.random.choice(N_PRE, size=N_PRE, replace=True)

    y_ca_boot = y_ca_pre[boot_idx]
    Y_donors_boot = Y_donors_pre[boot_idx, :]

    # Solve weights on bootstrap sample
    res_boot = minimize(
        objective,
        w0,
        args=(y_ca_boot, Y_donors_boot),
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 500, "ftol": 1e-9},
    )
    w_boot = res_boot.x

    # Post-period gap with bootstrap weights
    y_syn_post_boot = Y_donors_post @ w_boot
    bootstrap_gaps[b, :] = y_california[N_PRE:] - y_syn_post_boot

# Compute percentile intervals
gap_lower = np.percentile(bootstrap_gaps, 3, axis=0)
gap_upper = np.percentile(bootstrap_gaps, 97, axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(post_years, gap_lower, gap_upper, alpha=0.25, color="#e74c3c",
                label="94% Bootstrap CI")
ax.plot(post_years, gap_post, color="#e74c3c", linewidth=2.5, label="Estimated gap")
ax.plot(post_years, true_effects, color="#27ae60", linewidth=2, linestyle=":",
        label="True effect")
ax.axhline(0, color="black", linestyle="--", linewidth=1.5)
ax.set_title("Causal Effect Estimate with Bootstrap Uncertainty", fontsize=13)
ax.set_xlabel("Year (post-intervention)")
ax.set_ylabel("Gap (packs per capita per year)")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Note: Bootstrap uncertainty captures weight estimation uncertainty only.")
print("Permutation tests (Notebook 02) provide the valid statistical inference.")

## Summary

### What We Covered

1. **Panel dataset structure:** One treated unit (California), multiple donor units, pre- and post-intervention periods
2. **Weight optimization:** Constrained quadratic programming via scipy.optimize.minimize with SLSQP
3. **Pre-period fit:** RMSPE diagnostic — the synthetic control is credible when RMSPE is small relative to outcome variation
4. **Counterfactual gap:** The difference between observed California and Synthetic California estimates the causal effect
5. **Weight sparsity:** Most donors receive zero weight — only a few comparable states contribute

### Key Numbers

- The optimization recovered weights close to the true weights (Colorado ~40%, Utah ~35%, Montana ~25%)
- Pre-period RMSPE is small, validating the synthetic control
- Post-period gap closely tracks the true treatment effect

### What's Next

**Notebook 02:** Placebo Tests — the valid statistical inference method for synthetic control,
using permutation of the treatment across donor units to construct a null distribution.